This project fine-tunes Qwen2.5-1.5B-Instruct into a domain-specific Indian cooking assistant called ChefMate. A 76-example ChatML cooking dataset is used for 4-bit QLoRA/LoRA fine-tuning with Unsloth and TRL. The trained model is exported to GGUF format and deployed locally using Ollama for Indian cooking queries. The model is evaluated against the base model and previous fine-tuned versions using five cooking questions.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
%pip install -q "unsloth[colab-new]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

In [ ]:
import unsloth

print("Unsloth installed successfully!")
print("Unsloth version:", unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully!
Unsloth version: 2026.9.14


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving chatml_cooking_dataset_v2.json to chatml_cooking_dataset_v2.json


In [ ]:
import json

DATASET_FILE = "chatml_cooking_dataset_v2.json"

with open(DATASET_FILE, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Dataset loaded successfully!")
print("Total examples:", len(data))

print("\nFirst example:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Dataset loaded successfully!
Total examples: 76

First example:
{
  "messages": [
    {
      "role": "user",
      "content": "How do I prepare dal tadka?\nI want to cook a simple Indian dish for my family."
    },
    {
      "role": "assistant",
      "content": "Here's a simple recipe for dal tadka: Boil 1 cup of dried red lentils (masoor dal) in 2 cups of water for about 15 minutes until they are soft. Drain and set aside. In the same pot, heat 2 tablespoons of vegetable oil over medium heat. Add 1 teaspoon cumin seeds and let them splutter for a few seconds. Add 1 finely chopped onion, 2 minced cloves of garlic, 1 grated ginger-sized piece, and sauté until the onions are golden. Stir in 1 teaspoon turmeric powder, 1/2 teaspoon cayenne pepper, and 1 teaspoon garam masala. Mix in the cooked dal and 1 cup of water, and cook for another 5 minutes until everything is well combined. Serve hot with rice or roti."
    }
  ]
}


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("Model loaded successfully!")
print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
Max sequence length: 2048


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(data)

print("Hugging Face Dataset created!")
print("Number of examples:", len(train_dataset))
print("\nColumns:", train_dataset.column_names)
print("\nFirst example:")
print(train_dataset[0])

Hugging Face Dataset created!
Number of examples: 76

Columns: ['messages']

First example:
{'messages': [{'content': 'How do I prepare dal tadka?\nI want to cook a simple Indian dish for my family.', 'role': 'user'}, {'content': "Here's a simple recipe for dal tadka: Boil 1 cup of dried red lentils (masoor dal) in 2 cups of water for about 15 minutes until they are soft. Drain and set aside. In the same pot, heat 2 tablespoons of vegetable oil over medium heat. Add 1 teaspoon cumin seeds and let them splutter for a few seconds. Add 1 finely chopped onion, 2 minced cloves of garlic, 1 grated ginger-sized piece, and sauté until the onions are golden. Stir in 1 teaspoon turmeric powder, 1/2 teaspoon cayenne pepper, and 1 teaspoon garam masala. Mix in the cooked dal and 1 cup of water, and cook for another 5 minutes until everything is well combined. Serve hot with rice or roti.", 'role': 'assistant'}]}


In [ ]:
def format_chat_template(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }


formatted_dataset = train_dataset.map(format_chat_template)

print("Chat template applied successfully!")
print("Number of examples:", len(formatted_dataset))

print("\nFormatted first example:")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/76 [00:00<?, ? examples/s]

Chat template applied successfully!
Number of examples: 76

Formatted first example:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I prepare dal tadka?
I want to cook a simple Indian dish for my family.<|im_end|>
<|im_start|>assistant
Here's a simple recipe for dal tadka: Boil 1 cup of dried red lentils (masoor dal) in 2 cups of water for about 15 minutes until they are soft. Drain and set aside. In the same pot, heat 2 tablespoons of vegetable oil over medium heat. Add 1 teaspoon cumin seeds and let them splutter for a few seconds. Add 1 finely chopped onion, 2 minced cloves of garlic, 1 grated ginger-sized piece, and sauté until the onions are golden. Stir in 1 teaspoon turmeric powder, 1/2 teaspoon cayenne pepper, and 1 teaspoon garam masala. Mix in the cooked dal and 1 cup of water, and cook for another 5 minutes until everything is well combined. Serve hot with rice or roti.<|im_end|>



In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./cooking_qwen_lora_v2",

    # Assignment requirement
    max_steps=100,

    # Training setup
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=2e-4,
    logging_steps=10,

    # Optimizer
    optim="adamw_8bit",

    # Regularization
    weight_decay=0.01,
    warmup_steps=5,

    # T4 GPU
    fp16=True,
    bf16=False,

    # Logging and checkpoints
    save_steps=50,
    save_total_limit=2,

    # Sequence length
    max_length=2048,

    # Dataset
    dataset_text_field="text",
    packing=False,

    seed=3407,
)

print("Training configuration created successfully!")
print("Max steps:", training_args.max_steps)
print("Learning rate:", training_args.learning_rate)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Logging steps:", training_args.logging_steps)

Training configuration created successfully!
Max steps: 100
Learning rate: 0.0002
Batch size: 2
Gradient accumulation: 4
Logging steps: 10


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    args=training_args,
)

print("SFTTrainer created successfully!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/76 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFTTrainer created successfully!


In [ ]:
train_result = trainer.train()

print("\nTraining completed successfully!")
print("Training loss:", train_result.training_loss)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 76 | Num Epochs = 10 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_avai

Step,Training Loss
10,2.152368
20,1.293647
30,1.084258
40,0.904149
50,0.743752
60,0.597198
70,0.455874
80,0.344664
90,0.260209
100,0.210426


Unsloth: Restored added_tokens_decoder metadata in ./cooking_qwen_lora_v2/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./cooking_qwen_lora_v2/checkpoint-100/tokenizer_config.json.



Training completed successfully!
Training loss: 0.8046546745300293


In [ ]:
ADAPTER_PATH = "./cooking_qwen_lora_v2_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("LoRA adapter saved successfully!")
print("Location:", ADAPTER_PATH)

Unsloth: Restored added_tokens_decoder metadata in ./cooking_qwen_lora_v2_adapter/tokenizer_config.json.


LoRA adapter saved successfully!
Location: ./cooking_qwen_lora_v2_adapter


In [ ]:
MERGED_PATH = "./cooking_qwen_merged_v2"

model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit",
)

print("Merged model saved successfully!")
print("Location:", MERGED_PATH)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ./cooking_qwen_merged_v2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:31<00:00, 31.14s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:38<00:00, 38.34s/it]


Unsloth: Merge process complete. Saved to `/content/cooking_qwen_merged_v2`
Merged model saved successfully!
Location: ./cooking_qwen_merged_v2


In [ ]:
GGUF_PATH = "./cooking_qwen_gguf_v2"

model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF export completed successfully!")
print("Location:", GGUF_PATH)

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in cooking_qwen_gguf_v2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:42<00:00, 42.69s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:37<00:00, 37.51s/it]


Unsloth: Merge process complete. Saved to `/content/cooking_qwen_gguf_v2`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11160-mix-a6922cc (app-b11160-mix-a6922cc-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['cooking_qwen_gguf_v2_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversion

In [ ]:
from google.colab import files

GGUF_FILE = "./cooking_qwen_gguf_v2_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf"

files.download(GGUF_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>